# PostgreSQL Transactions and Lock Diagnosis

[![Open Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Download this notebook, open Colab, and choose **File > Upload notebook**. The full draft course is distributed separately from the public Week 1 repository.


This notebook creates one controlled blocking relationship in a disposable schema,
identifies the blocked and blocking sessions, resolves the blocker, and verifies
the final row.

**The central idea:** a wait is a relationship between database sessions. Diagnose
that relationship before terminating or changing anything.

The cloud path asks for a temporary PostgreSQL URL with `getpass`, so the value is
not displayed or written into the notebook. A labeled transcript supports
interpretation when a connection is unavailable; it does not execute PostgreSQL.

## Before You Connect

1. Use a personal course database, never a production system.
2. In Supabase's **Connect** dialog, choose **Session pooler** for this exercise.
   It supports IPv4 and keeps each connection attached to a database session.
   Do not substitute the transaction-pooler endpoint: we are observing sessions.
3. Rotate the temporary password after class if required by your course policy.
4. Never paste a URL into a code or Markdown cell.

The URL must request `sslmode=require`, `verify-ca`, or `verify-full`; the
notebook rejects libpq's default `prefer` mode because it can fall back to an
unencrypted connection. For production, use `verify-full` with the Supabase CA
certificate. `require` encrypts the classroom connection but does not verify the
hostname. With a configured root certificate, libpq's `require` mode can also
check the CA; use `verify-full` when both CA and hostname verification are required.

[Supabase connection methods](https://supabase.com/docs/guides/database/connecting-to-postgres)

Set `USE_CLOUD` to `True` only when you are ready. It remains `False` in the public
notebook so all non-cloud cells can run safely without credentials.

In [ ]:
%pip -q install "psycopg[binary]"

In [ ]:
from getpass import getpass
import threading
import time

import psycopg
from psycopg.conninfo import conninfo_to_dict

USE_CLOUD = False  # Change to True during the in-class cloud lab.
KEEP_A_CHANGE = False  # First run: rollback A. Second run: set True to commit A.
print("Cloud path enabled:", USE_CLOUD)

## 1. Open Three Clearly Named Sessions

- **Session A** will update a row and deliberately remain uncommitted.
- **Session B** will attempt a competing update and wait.
- **Diagnostic session** will query PostgreSQL's activity and lock state.

Three connections make the roles visible. The diagnostic session does not cause
or resolve the block; it observes it.

In [ ]:
if USE_CLOUD:
    database_url = getpass("Paste the temporary PostgreSQL connection URL: ")

    try:
        sslmode = conninfo_to_dict(database_url).get("sslmode", "prefer")
    except psycopg.Error:
        database_url = None
        raise ValueError("The connection URL could not be parsed. Copy a fresh Session pooler URL.") from None
    if sslmode not in {"require", "verify-ca", "verify-full"}:
        database_url = None
        raise ValueError("Add sslmode=require or a stronger mode to the temporary URL.")

    session_a = session_b = diagnostic = None
    try:
        session_a = psycopg.connect(
            database_url, application_name="cst4714_session_a", connect_timeout=10
        )
        session_b = psycopg.connect(
            database_url, application_name="cst4714_session_b", connect_timeout=10
        )
        diagnostic = psycopg.connect(
            database_url, application_name="cst4714_diagnostic", connect_timeout=10
        )
        diagnostic.autocommit = True
    except psycopg.Error:
        for connection in (session_a, session_b, diagnostic):
            if connection is not None:
                connection.close()
        raise RuntimeError("Connection failed. Check project status, Session pooler URL, password, and SSL mode.") from None
    finally:
        database_url = None

    print("Opened Session A, Session B, and the diagnostic session.")
else:
    print("Cloud path skipped. Continue to the fallback transcript below.")

## 2. Create a Disposable Target

The table has one row. Its original state is `priority = medium` and
`status = open`. This deliberately simplified fixture differs from the full
Metro Support ticket. Recreating `lock_lab` resets only this exercise.

In [ ]:
if USE_CLOUD:
    with diagnostic.cursor() as cursor:
        cursor.execute("DROP SCHEMA IF EXISTS lock_lab CASCADE")
        cursor.execute("CREATE SCHEMA lock_lab")
        cursor.execute("""
            CREATE TABLE lock_lab.tickets (
                ticket_id integer PRIMARY KEY,
                priority text NOT NULL,
                status text NOT NULL
            )
        """)
        cursor.execute("""
            INSERT INTO lock_lab.tickets (ticket_id, priority, status)
            VALUES (1004, 'medium', 'open')
        """)
        cursor.execute("SELECT * FROM lock_lab.tickets")
        print("Starting row:", cursor.fetchone())
else:
    print("Setup skipped because USE_CLOUD is False.")

## 3. Session A Holds an Uncommitted Row Change

Session A will change the priority without committing. Read this statement and
predict what a second writer will do. The next experiment runs both sessions,
captures the waiting relationship, and releases the lock in the same cell. You
can study its saved results without leaving a database transaction open.

For the first run, `KEEP_A_CHANGE = False` discards A's proposed priority change.
After cleanup, change it to `True` and run again from the configuration cell.
Predict the second final row before running it. B's SQL stays the same.

In [ ]:
session_a_sql = """
    UPDATE lock_lab.tickets
    SET priority = 'high'
    WHERE ticket_id = 1004
    RETURNING ticket_id, priority, status
"""
print("Session A will run this UPDATE without committing:")
print(session_a_sql)

## 4. Session B Starts a Competing Update

The notebook uses one small background thread because a blocked query cannot both
wait and let the same notebook cell collect diagnostics. The experiment captures
the real PostgreSQL lock relationship, ends A's transaction, and lets B commit
before the cell ends. `KEEP_A_CHANGE` chooses A's outcome. The `finally` block
rolls back any remaining transaction if the experiment fails.

You do not need to write threading code for this lab. Follow the SQL and the
transaction decisions. The small worker below allows B to wait while another
connection runs the diagnostic query.

In [ ]:
blocked_result = {}

def run_session_b_update():
    try:
        with session_b.cursor() as cursor:
            cursor.execute("SET statement_timeout = '15s'")
            cursor.execute("""
                UPDATE lock_lab.tickets
                SET status = 'in_progress'
                WHERE ticket_id = 1004
                RETURNING ticket_id, priority, status
            """)
            blocked_result["row"] = cursor.fetchone()
        session_b.commit()
        blocked_result["outcome"] = "committed after blocker released"
    except Exception as error:
        session_b.rollback()
        blocked_result["outcome"] = f"error: {type(error).__name__}"


diagnostic_sql = """
    SELECT pid, application_name, state, wait_event_type, wait_event,
           pg_blocking_pids(pid) AS blocking_pids,
           xact_start, clock_timestamp() - xact_start AS transaction_age,
           left(query, 90) AS query_sample
    FROM pg_stat_activity
    WHERE pid IN (%s, %s)
    ORDER BY application_name
"""
rows = []
columns = []
session_b_thread = None

if USE_CLOUD:
    try:
        with session_a.cursor() as cursor:
            cursor.execute("SET idle_in_transaction_session_timeout = '30s'")
            cursor.execute("SELECT pg_backend_pid()")
            a_pid = cursor.fetchone()[0]
            cursor.execute(session_a_sql)
            session_a_row = cursor.fetchone()
        # Psycopg began a transaction. A has not committed.
        with diagnostic.cursor() as cursor:
            cursor.execute("SELECT ticket_id, priority, status FROM lock_lab.tickets")
            reader_row = cursor.fetchone()
        with session_b.cursor() as cursor:
            cursor.execute("SELECT pg_backend_pid()")
            b_pid = cursor.fetchone()[0]
        # Ask PostgreSQL itself for PIDs, rather than a proxy's protocol identity.
        session_pids = (a_pid, b_pid)
        session_b_thread = threading.Thread(target=run_session_b_update)
        session_b_thread.start()

        # Capture the real wait instead of assuming it began after a fixed sleep.
        for attempt in range(100):
            with diagnostic.cursor() as cursor:
                cursor.execute(diagnostic_sql, session_pids)
                columns = [column.name for column in cursor.description]
                rows = cursor.fetchall()
            if any(row[0] == session_pids[1] and session_pids[0] in row[5] for row in rows):
                break
            time.sleep(0.05)
        else:
            raise RuntimeError("No blocking relationship was captured; rerun from setup.")
        if KEEP_A_CHANGE:
            session_a.commit()
        else:
            session_a.rollback()
    finally:
        # Always release A, even when diagnosis raises an error.
        session_a.rollback()
        if session_b_thread is not None:
            session_b_thread.join(timeout=20)
    if session_b_thread is not None and session_b_thread.is_alive():
        session_b.cancel()
        session_b_thread.join(timeout=5)
        raise RuntimeError("Session B did not finish; close the connections and rerun.")

    with diagnostic.cursor() as cursor:
        cursor.execute("SELECT ticket_id, priority, status FROM lock_lab.tickets")
        final_row = cursor.fetchone()
    assert blocked_result.get("outcome") == "committed after blocker released", blocked_result
    expected_priority = "high" if KEEP_A_CHANGE else "medium"
    assert final_row == (1004, expected_priority, "in_progress"), final_row
    print("A saw its uncommitted change:", session_a_row)
    print("An ordinary reader still saw:", reader_row)
    print("Captured a waiting writer, released A, and verified B's commit.")
    print("The next sections inspect the saved results. No lab row lock remains.")
else:
    print("Experiment skipped. Use the supplied transcript below.")

## 5. Ask PostgreSQL Who Is Blocking Whom

`pg_blocking_pids(pid)` directly reports the blocker relationship. The
wait event adds context. A PID identifies a session; it is not automatic
permission to terminate a process.

The query now selects only the two PIDs opened by this run. `%s` marks a value
supplied separately by Psycopg; it is not SQL you should replace by string
concatenation. `transaction_age` is measured at capture time. The waiter can
have `state = active` and still have `wait_event_type = Lock`.

In [ ]:
if USE_CLOUD:
    print("Snapshot captured while B was waiting:")
    for row in rows:
        print(f"\nSession {row[1]} (PID {row[0]})")
        for label, value in zip(columns[2:], row[2:]):
            print(f"  {label}: {value}")
else:
    print("Read the blocked and blocking PIDs in the fallback transcript.")

## 6. Resolve the Blocker and Verify the Final State

Both choices end A's transaction and release the row lock. Rollback discards its
priority change; commit keeps it. B then commits `status = in_progress` in either
case. Inspect both columns, not just whether B finished.

In [ ]:
if USE_CLOUD:
    print("A's resolution:", "COMMIT" if KEEP_A_CHANGE else "ROLLBACK")
    print("Session B outcome:", blocked_result)
    print("Final row captured from a fresh statement:", final_row)
else:
    print("Compare A's discarded priority change with B's committed status change.")

## 7. Close Connections and Remove the Disposable Schema

Cleanup is part of the operation. It prevents an old practice lock or test table
from becoming a later mystery.

In [ ]:
if USE_CLOUD:
    with diagnostic.cursor() as cursor:
        cursor.execute("DROP SCHEMA IF EXISTS lock_lab CASCADE")

    session_a.close()
    session_b.close()
    diagnostic.close()
    database_url = None
    print("Closed all sessions and removed lock_lab.")
else:
    print("No cloud resources were opened.")

## Offline Fallback Incident Transcript

Use this illustrative transcript when the cloud path cannot run. These are
teaching values, not measurements from your notebook. It represents the rollback
case. Predict the commit case separately; label that result as a prediction.

```text
Starting row: (1004, 'medium', 'open')

Session A PID 7310, application cst4714_session_a
Transaction began: 2026-03-05 18:42:10+00
Uncommitted query: UPDATE lock_lab.tickets SET priority = 'high'
                   WHERE ticket_id = 1004

Session B PID 7332, application cst4714_session_b
State: active
wait_event_type: Lock
wait_event: transactionid
pg_blocking_pids(7332): {7310}
Query: UPDATE lock_lab.tickets SET status = 'in_progress'
       WHERE ticket_id = 1004

Action: ROLLBACK issued in Session A.
Session B outcome: committed after blocker released.
Final row: (1004, 'medium', 'in_progress')
```

The transcript records one blocking relationship, the chosen resolution, and the
final state. It does not determine how a production application should choose
between waiting, rollback, cancellation, or termination.

## Your Comparison and Incident Update

Before the second run, keep the first final row here and predict the second.
After running again, replace the second observation with your result.

| A's decision | Predicted final priority/status | Observed final priority/status |
|---|---|---|
| ROLLBACK | Your prediction | Your result |
| COMMIT | Your prediction | Your result |

In the same cell, write a short update to the developer whose status update
appeared stuck. Use one run's PIDs and `pg_blocking_pids` result to explain the
wait, then explain why both runs let B finish but kept different priority values.
Name one reason you would not make the same commit/rollback decision blindly on
a real application. No separate incident form or report is required.

If you used the transcript, label the first row **supplied trace** and the second
**unexecuted prediction**. Do not describe that path as a live connection test.

Before submitting, remove any accidentally saved credential from source or output.

**License:** prose CC BY-NC-SA 4.0; code MIT.